# STEP 9: LLM Router로 실제 OpenDART workflow 실행

자연어 질문을 LLM Router가 구조화하고, LangGraph가 실제 OpenDART 조회와 계산 경로를 실행합니다.

질문 → LLM Router → 기업명·연도·분석 유형 State → OpenDART 조회 → 조건 분기 → 요약

이번 단계는 두 종류의 정상 질문만 다루는 학습용 workflow입니다. 복수 기업 비교, 누락 입력 처리, 재시도, Tool Calling, Agent, 최종 자연어 답변 생성은 아직 넣지 않습니다.

## 0. 경로와 역할

- basic: 매출액·영업이익·당기순이익을 조회하고 요약
- margin: 같은 재무정보를 조회한 뒤 영업이익률을 계산하고 요약

LLM은 질문을 구조화하는 Router 역할만 합니다. OpenDART 조회와 수치 계산은 Python Node가 담당합니다.

## 1. 프로젝트 경로와 API 키 준비

OpenDART API 키는 .env에서만 읽고, LLM·State·출력에는 넣지 않습니다.

In [1]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

current_dir = Path.cwd().resolve()
if current_dir.name == "learning":
    project_root = current_dir.parents[1]
elif current_dir.name == "notebooks":
    project_root = current_dir.parent
else:
    project_root = current_dir
sys.path.insert(0, str(project_root))
load_dotenv(project_root / ".env")
api_key = os.getenv("OPENDART_API_KEY")

if not api_key or api_key == "your_opendart_api_key":
    raise ValueError(".env의 OPENDART_API_KEY를 실제 API 키로 바꿔 주세요.")

print("프로젝트 경로와 OpenDART API 키를 준비했습니다.")

프로젝트 경로와 OpenDART API 키를 준비했습니다.


## 2. 필요한 클래스와 기존 OpenDART 함수 불러오기

OpenDART 호출 로직은 client.py에 이미 검증되어 있습니다. 이 노트북은 이를 import하고, LLM Router와 LangGraph 연결만 확인합니다.

In [2]:
from typing import Literal, TypedDict

from langchain_core.messages import HumanMessage, SystemMessage
from langchain_ollama import ChatOllama
from langgraph.graph import END, START, StateGraph
from pydantic import BaseModel, Field

from src.dart.client import get_financial_summary

  from .autonotebook import tqdm as notebook_tqdm


## 3. LLM Router의 구조화된 출력

이 notebook은 기업명과 4자리 사업연도가 질문에 명시된 정상 입력만 다룹니다. LLM이 이 세 값을 구조화해 반환하면 다음 Node들이 자연어를 다시 해석하지 않아도 됩니다.

In [3]:
class FinancialRequest(BaseModel):
    company_name: str = Field(description="질문에 나온 OpenDART 조회 대상 기업명")
    business_year: int = Field(
        ge=1000, le=9999, description="질문에 명시된 4자리 사업연도"
    )
    analysis_type: Literal["basic", "margin"] = Field(
        description="basic은 기본 재무 수치 조회, margin은 영업이익률 계산 또는 분석"
    )


llm = ChatOllama(
    model="qwen3.6:27b",
    base_url="http://localhost:11434",
    temperature=0,
)
structured_router = llm.with_structured_output(FinancialRequest)

## 4. State와 LLM Router Node

route_question Node가 질문에서 기업명, 사업연도, 분석 유형을 추출해 State에 추가합니다. API 키는 이 Node에 전달되지 않습니다.

In [4]:
class FinancialWorkflowState(TypedDict, total=False):
    question: str
    company_name: str
    business_year: int
    analysis_type: Literal["basic", "margin"]
    financial_data: dict[str, str | int | None]
    operating_margin: float
    summary: str


def route_question(state: FinancialWorkflowState) -> dict:
    messages = [
        SystemMessage(
            content=(
                "기업 재무 질문에서 기업명, 4자리 사업연도, 분석 유형을 추출한다. "
                "매출액, 영업이익, 당기순이익 같은 기본 재무 수치 조회는 basic이다. "
                "영업이익률의 계산 또는 분석은 margin이다. 두 종류가 함께 있으면 margin을 선택한다."
            )
        ),
        HumanMessage(content=state["question"]),
    ]
    request = structured_router.invoke(messages)
    print("LLM Router 결과:", request.model_dump())
    return request.model_dump()

## 5. 실제 OpenDART 조회 Node

이 Node는 Router가 State에 넣은 기업명과 사업연도를 사용합니다. LLM은 재무 수치를 생성하지 않고, 실제 OpenDART 조회 결과만 State에 추가합니다.

In [5]:
def fetch_financials(
    state: FinancialWorkflowState,
) -> dict[str, dict[str, str | int | None]]:
    financial_data = get_financial_summary(
        corp_name=state["company_name"],
        business_year=state["business_year"],
        api_key=api_key,
    )
    return {"financial_data": financial_data}

## 6. 조건 함수와 두 분석 경로 Node

조회가 끝난 뒤 Conditional Edge가 analysis_type을 읽습니다. basic은 세 재무 수치를 요약하고, margin은 영업이익률을 계산한 뒤 요약합니다.

In [6]:
def select_analysis_path(
    state: FinancialWorkflowState,
) -> Literal["basic_summary", "calculate_margin"]:
    if state["analysis_type"] == "basic":
        return "basic_summary"
    return "calculate_margin"


def display_value(value: str | int | None) -> str:
    return str(value) if value is not None else "조회되지 않음"


def create_basic_summary(state: FinancialWorkflowState) -> dict[str, str]:
    data = state["financial_data"]
    summary = (
        f'{data["기업명"]} {data["사업연도"]}년 재무정보\n'
        f'- 매출액: {display_value(data["매출액"])}\n'
        f'- 영업이익: {display_value(data["영업이익"])}\n'
        f'- 당기순이익: {display_value(data["당기순이익"])}'
    )
    return {"summary": summary}


def calculate_operating_margin(state: FinancialWorkflowState) -> dict[str, float]:
    data = state["financial_data"]
    revenue = data["매출액"]
    operating_income = data["영업이익"]
    if revenue is None or operating_income is None:
        raise ValueError("영업이익률 계산에 필요한 재무 수치가 없습니다.")
    margin = int(operating_income) / int(revenue) * 100
    return {"operating_margin": round(margin, 2)}


def create_margin_summary(state: FinancialWorkflowState) -> dict[str, str]:
    data = state["financial_data"]
    summary = (
        f'{data["기업명"]} {data["사업연도"]}년 영업이익률\n'
        f'- 매출액: {display_value(data["매출액"])}\n'
        f'- 영업이익: {display_value(data["영업이익"])}\n'
        f'- 영업이익률: {state["operating_margin"]}%'
    )
    return {"summary": summary}

## 7. 전체 Graph 연결

모든 질문은 먼저 LLM Router와 실제 OpenDART 조회를 거칩니다. 그 뒤에만 analysis_type에 따라 basic 또는 margin 경로로 나뉩니다.

In [7]:
workflow = StateGraph(FinancialWorkflowState)

workflow.add_node("route_question", route_question)
workflow.add_node("fetch_financials", fetch_financials)
workflow.add_node("basic_summary", create_basic_summary)
workflow.add_node("calculate_margin", calculate_operating_margin)
workflow.add_node("margin_summary", create_margin_summary)

workflow.add_edge(START, "route_question")
workflow.add_edge("route_question", "fetch_financials")
workflow.add_conditional_edges(
    "fetch_financials",
    select_analysis_path,
    {
        "basic_summary": "basic_summary",
        "calculate_margin": "calculate_margin",
    },
)
workflow.add_edge("basic_summary", END)
workflow.add_edge("calculate_margin", "margin_summary")
workflow.add_edge("margin_summary", END)

app = workflow.compile()

## 8. Graph 구조 확인

Mermaid 텍스트에서 Router 다음에 OpenDART 조회가 있고, 그 다음에 조건 분기가 있는지 확인합니다.

In [8]:
print(app.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	route_question(route_question)
	fetch_financials(fetch_financials)
	basic_summary(basic_summary)
	calculate_margin(calculate_margin)
	margin_summary(margin_summary)
	__end__([<p>__end__</p>]):::last
	__start__ --> route_question;
	calculate_margin --> margin_summary;
	fetch_financials -.-> basic_summary;
	fetch_financials -.-> calculate_margin;
	route_question --> fetch_financials;
	basic_summary --> __end__;
	margin_summary --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## 9. basic 질문 실행

LLM Router가 기업명, 연도, basic을 구조화하면 OpenDART 조회 뒤 basic_summary Node가 실행됩니다.

In [9]:
basic_question = "삼성전자의 2025년 매출액과 영업이익, 당기순이익을 알려줘."
basic_result = app.invoke({"question": basic_question})
print(basic_result["summary"])

LLM Router 결과: {'company_name': '삼성전자', 'business_year': 2025, 'analysis_type': 'basic'}
삼성전자 2025년 재무정보
- 매출액: 333605938000000
- 영업이익: 43601051000000
- 당기순이익: 45206805000000


## 10. margin 질문 실행

LLM Router가 margin을 반환하면 OpenDART 조회 뒤 calculate_margin Node와 margin_summary Node가 실행됩니다.

In [10]:
margin_question = "삼성전자의 2025년 영업이익률을 분석해줘."
margin_result = app.invoke({"question": margin_question})
print(margin_result["summary"])

LLM Router 결과: {'company_name': '삼성전자', 'business_year': 2025, 'analysis_type': 'margin'}
삼성전자 2025년 영업이익률
- 매출액: 333605938000000
- 영업이익: 43601051000000
- 영업이익률: 13.07%


## 11. State에서 Router 결과와 실제 데이터 확인

두 실행 결과에서 LLM이 만든 구조화 값과 실제 OpenDART 데이터가 같은 State에 저장됐는지 확인합니다.

In [11]:
print("basic Router 결과:", {
    key: basic_result[key]
    for key in ("company_name", "business_year", "analysis_type")
})
print("margin Router 결과:", {
    key: margin_result[key]
    for key in ("company_name", "business_year", "analysis_type")
})
print("margin OpenDART 데이터:", margin_result["financial_data"])

basic Router 결과: {'company_name': '삼성전자', 'business_year': 2025, 'analysis_type': 'basic'}
margin Router 결과: {'company_name': '삼성전자', 'business_year': 2025, 'analysis_type': 'margin'}
margin OpenDART 데이터: {'기업명': '삼성전자', '사업연도': 2025, '매출액': '333605938000000', '영업이익': '43601051000000', '당기순이익': '45206805000000'}


## 12. 최소 확인

이 확인은 정량 평가가 아니라, 두 정상 질문이 Router, OpenDART Node, Conditional Edge, 계산 Node까지 연결되는지 보는 학습용 점검입니다.

In [12]:
assert basic_result["company_name"] == "삼성전자"
assert basic_result["business_year"] == 2025
assert basic_result["analysis_type"] == "basic"
assert margin_result["analysis_type"] == "margin"
assert margin_result["operating_margin"] > 0

print("LLM Router 결과가 실제 OpenDART 조회와 분석 경로에 연결되었습니다.")

LLM Router 결과가 실제 OpenDART 조회와 분석 경로에 연결되었습니다.


## 이번 단계에서 기억할 점

- LLM Router는 자연어에서 기업명·연도·분석 유형을 구조화합니다.
- OpenDART Node는 LLM이 만든 인자를 사용하지만, 수치는 실제 API에서만 가져옵니다.
- Conditional Edge는 LLM Router가 만든 analysis_type을 보고 계산 경로를 선택합니다.
- 이 예제는 정상 입력 두 종류만 다룹니다. Router 오분류, 기업명 불일치, 누락 연도, API 오류, 재시도는 이후 별도 학습과 검증 대상입니다.